In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.signal import find_peaks, hilbert
from scipy.stats import circmean
from PIL import Image

from ruler_confidence import row_spectrum, group_evidence, ruler_confidence

import warnings
warnings.filterwarnings('ignore')

In [ ]:

#IMAGE_PATH = "../annotated_images/luomus_pictures_1/8/F.8368.jpg"  
#IMAGE_PATH =  "F.82978.jpg"
IMAGE_PATH = "Amegilla_fasciata_M_001.jpg"

GRADUATION_MM = 1.0 

MIN_FREQ_RATIO = 0.005        
MAX_FREQ_RATIO = 0.1    
PEAK_PROMINENCE = 0.1 

RATIO = 5

N_LINES = None        

print(f"Expected gradation : {GRADUATION_MM} mm")
print(f"Valid period spectrum : [{MIN_FREQ_RATIO} – {MAX_FREQ_RATIO}]")

In [ ]:
def load_image(path: str) -> tuple[np.ndarray, np.ndarray]:
    img = Image.open(path).convert("RGB")
    img_color = np.array(img)

    img_gray = np.dot(img_color[..., :3], [0.299, 0.587, 0.114]).astype(np.float32) / 255.0

    print(f"Image size : {img_gray.shape[1]} × {img_gray.shape[0]} px")
    return img_gray, img_color


base_img_gray, base_img_color = load_image(IMAGE_PATH)

ruler_ratio = 0

img_gray = base_img_gray[int(ruler_ratio*base_img_gray.shape[0]):, :]
img_color = base_img_color[int(ruler_ratio*base_img_color.shape[0]):, :, :]
H, W = img_gray.shape

Ho, Wo = (500, 500)

reduced_img_gray = img_gray[::RATIO, ::RATIO]
Hm, Wm = reduced_img_gray.shape

fig, ax = plt.subplots(figsize=(12, 5))
ax.imshow(reduced_img_gray)
ax.set_title(f"Original image : {Wm}×{Hm} px (vs {W}x{H})", fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()


H, W = reduced_img_gray.shape

In [ ]:
def fft_dominant_frequency(row: np.ndarray,
                            min_period_ratio: float,
                            max_period_ratio: float,
                            prominence: float) -> tuple[float | None, float | None]:
    """
    Apply the FFT to a row of pixels and return the dominant period (in px) and its
    phase.

    Parameters
    ----------
    row           : 1D array of grey levels
    min_period_px : minimum accepted period (px)
    max_period_px : maximum accepted period (px)
    prominence    : minimum relative prominence of the peak

    Returns
    -------
    (period_px, phase_rad) or (None, None) if no valid peak
    """
    N = len(row)

    row_centered = row - row.mean()

    window = np.hanning(N)
    row_windowed = row_centered * window

    fft_vals = np.fft.rfft(row_windowed)
    freqs    = np.fft.rfftfreq(N)   # normalised frequencies [0, 0.5]

    magnitude = np.abs(fft_vals)

    f_min = 1.0 / (max_period_ratio * N)
    f_max = 1.0 / (min_period_ratio * N)

    mask = (freqs >= f_min) & (freqs <= f_max)
    if mask.sum() == 0:
        return None, None

    mag_masked = magnitude.copy()
    mag_masked[~mask] = 0

    peaks, props = find_peaks(mag_masked,
                              prominence=prominence * mag_masked.max())
    if len(peaks) == 0:
        return None, None

    best_peak = peaks[np.argmax(mag_masked[peaks])]

    period_px = 1.0 / freqs[best_peak]
    phase_rad = np.angle(fft_vals[best_peak])

    return period_px, phase_rad, max(mag_masked[peaks])

def observed_cycles(row, period_px, env_threshold=0.2):
    N = len(row)
    f0 = 1.0 / period_px
    row_c = row - row.mean()                 # NO window here (the edges are kept)

    # band-pass around f0
    fft   = np.fft.rfft(row_c)
    freqs = np.fft.rfftfreq(N)
    bw    = 0.5 * f0                          # ±50 % around f0
    fft[(freqs < f0 - bw) | (freqs > f0 + bw)] = 0
    filtered = np.fft.irfft(fft, n=N)

    # amplitude envelope of the component
    env = np.abs(hilbert(filtered))
    threshold = env_threshold * env.max()

    # crests of the oscillation, only where it is actually present
    peaks, _ = find_peaks(filtered, distance=max(1, period_px * 0.5))
    peaks = peaks[env[peaks] >= threshold]
    return len(peaks)

row_indices = np.arange(H)
if N_LINES is not None and N_LINES < H:
    row_indices = np.linspace(0, H - 1, N_LINES, dtype=int)

# results = []   
# n_cycles = []

# for i in row_indices:
#     # row = img_gray[i, :]
#     row = reduced_img_gray[i, :]
#     period, phase, magnitude = fft_dominant_frequency(
#         row, MIN_FREQ_RATIO, MAX_FREQ_RATIO, PEAK_PROMINENCE
#     )
#     if period is not None:
#         results.append((i, period, phase, magnitude))
#         n_cycles.append(observed_cycles(row, period))

results, n_cycles, row_specs = [], [], []
for i in row_indices:
    row = reduced_img_gray[i, :]
    rs = row_spectrum(row, MIN_FREQ_RATIO, MAX_FREQ_RATIO, PEAK_PROMINENCE)
    if rs is not None:
        results.append((i, rs.period_px, rs.phase_rad, rs.magnitude))
        n_cycles.append(observed_cycles(row, rs.period_px))
        row_specs.append(rs)

print(n_cycles)

print(f"analysed lines: {len(row_indices)}")
print(f"valid peak lines: {len(results)} ({100*len(results)/len(row_indices):.1f}%)")

In [ ]:
# # speed test
# import time as time
# import tqdm

# ntry = H

# times = []

# for size in tqdm.tqdm(range(100, W, 200)):
#     S = 0 
#     test_row = row[:size]
#     for i in range(ntry):
#         T1 = time.time()
#         fft_dominant_frequency(test_row, MIN_FREQ_RATIO, MAX_FREQ_RATIO, PEAK_PROMINENCE)
#         T2 = time.time()
#         S += (T2 - T1)
#     times.append(S / ntry)

# print(list(range(100, W, 100)))       
# print(times)

In [ ]:
# X = np.array(list(range(100, W, 200)))
# Y = np.array(times) 

# m, b = np.polyfit(X, Y, 1)
# print(m, b) # says : 1.103616662614173e-07 0.0002810560613318752
# # -> gains 10-4 seconds per 1000 px (0.5s on a 5000 px height)

# plt.scatter(X, Y)
# plt.show()

In [ ]:

mag_max = 0
max_idx = 0
for i in range(len(results)):
    mid_row  = results[i][0]
    if results[i][3] > mag_max:
        mag_max = results[i][3]
        max_idx = i

mid_row = results[max_idx][0]
row_demo = img_gray[mid_row, :]
N        = len(row_demo)

row_c = row_demo - row_demo.mean()
row_w = row_c * np.hanning(N)
fft_demo  = np.fft.rfft(row_w)
freqs_demo = np.fft.rfftfreq(N)
mag_demo  = np.abs(fft_demo)

with np.errstate(divide='ignore'):
    periods_demo = np.where(freqs_demo > 0, 1.0 / freqs_demo, np.inf)

valid_mask = (periods_demo >= N * MIN_FREQ_RATIO) & (periods_demo <= N * MAX_FREQ_RATIO)

fig, axes = plt.subplots(2, 1, figsize=(13, 6))

axes[0].plot(row_demo, color='steelblue', lw=0.7)
axes[0].set_title(f"Pixel intensity - line {mid_row}", fontsize=11)
axes[0].set_xlabel("x position (px)")
axes[0].set_ylabel("normalized intensity")
axes[0].grid(True, alpha=0.3)

axes[1].plot(periods_demo[valid_mask], mag_demo[valid_mask],
             color='darkorange', lw=1.2, label='Spectrum (valid range)')
axes[1].plot(periods_demo[~valid_mask & (freqs_demo > 0)],
             mag_demo[~valid_mask & (freqs_demo > 0)],
             color='gray', lw=0.6, alpha=0.4, label='Out of range')

if len(results) > 0:
    dominant_period = periods_demo[valid_mask][np.argmax(mag_demo[valid_mask])]
    print(f"Max magnitude : {mag_demo[valid_mask].max():.2f}")
    print(f"dominant period : {dominant_period:.2f} px -> {dominant_period:.2f} px / mm")
    axes[1].axvline(dominant_period, color='red', ls='--', lw=1.5,
                    label=f'median peak ≈ {dominant_period:.1f} px')

axes[1].set_title("FFT magnitude spectrum", fontsize=11)
axes[1].set_xlabel("Period (px/graduation)")
axes[1].set_ylabel("Magnitude")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
if len(results) < 5:
    raise RuntimeError("Error : not enough valid line")

rows_arr   = np.array([r[0] for r in results], dtype=float)
phases_arr  = np.array([r[2] for r in results], dtype=float)
mag_arr = np.array([r[3] for r in results], dtype=float)

periods_arr = np.array([r[1] for r in results], dtype=float)
max_mag = max(mag_arr)
filtered_rows = []
filtered_phases = []
filtered_periods = []
for i in range(len(results)):
    if results[i][3] == max_mag:
        filtered_rows.append(results[i][0])
        filtered_phases.append(results[i][2])
        filtered_periods.append(results[i][1])

filtered_rows_arr = np.array(filtered_rows, dtype=float)
filtered_phases_arr = np.array(filtered_phases, dtype=float)
filtered_periods_arr = np.array(filtered_periods, dtype=float)


T_median = np.median(filtered_periods_arr)
print(f"Rough median period : {T_median:.3f} px")

phases_unwrapped = np.unwrap(filtered_phases_arr)


slope_phase, intercept_phase = np.polyfit(filtered_rows_arr, phases_unwrapped, 1)

print(f"Phase slope (rad/px) : {slope_phase:.6f}")

tan_theta = slope_phase * T_median / (2 * np.pi)
theta_rad = np.arctan(tan_theta)
theta_deg = np.degrees(theta_rad)

# maximum explained variance per period for each row: (amp²/2) / var(row), must be ∈ [0, 1]
explain_var = []
for i in range(len(results)):
    row = reduced_img_gray[results[i][0], :]
    period = results[i][1]
    f0 = 1.0 / period
    row_c = row - row.mean()                 # NO window here (the edges are kept)

    # band-pass around f0
    fft   = np.fft.rfft(row_c)
    freqs = np.fft.rfftfreq(len(row_c))
    bw    = 0.5 * f0                          # ±50 % around f0
    fft[(freqs < f0 - bw) | (freqs > f0 + bw)] = 0
    filtered = np.fft.irfft(fft, n=len(row_c))

    # amplitude envelope of the component
    env = np.abs(hilbert(filtered))
    amp2 = (env**2).mean()
    var_row = row.var()
    if var_row > 0:
        explain_var.append(min(amp2 / (2 * var_row), 1.0))
    else:
        explain_var.append(0.0)

print(f"\nEstimated inclination angle : {theta_deg:.3f}°")
print(f"Explained variance : {max(explain_var):.3f}")

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

axes[0, 0].scatter(rows_arr, periods_arr, s=4, alpha=0.5, c=[res[3] for res in results], cmap='viridis',)
axes[0, 0].set_title("FFT period per pixel line", fontsize=11)
axes[0, 0].set_xlabel("Line y (px)")
axes[0, 0].set_ylabel("Detected period (px)")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

axes[1, 0].scatter(rows_arr, np.unwrap(phases_arr), s=4, alpha=0.5, c=[res[3] for res in results], cmap='viridis')
axes[1, 0].set_title("Phase drift between lines", fontsize=11)
axes[1, 0].set_xlabel("Line y (px)")
axes[1, 0].set_ylabel("Phase (rad)")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(rows_arr, n_cycles, s=4, alpha=0.5, c=[res[3] for res in results], cmap='viridis')
axes[0, 1].set_title("Number of detected cycles", fontsize=11)
axes[0, 1].set_xlabel("Line y (px)")
axes[0, 1].set_ylabel("n cycles")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

axes[1, 1].scatter(rows_arr, explain_var, s=4, alpha=0.5, c=[res[3] for res in results], cmap='viridis')
axes[1, 1].set_title("Explained Variance", fontsize=11)
axes[1, 1].set_xlabel("Line y (px)")
axes[1, 1].set_ylabel("Explained Variance")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np


def _initial_slope(seed, period, lines, order, rank, delta, win=3):
    """Estimate the local slope period=f(lines) around the seed, from the neighbouring
    points (on the 'lines' axis) that stay close in period.
    Useful when the crest is tilted and the seed has only one neighbour."""
    pos = rank[seed]
    window = 4.0 * delta
    idx = []
    for d in range(-win, win + 1):
        p = pos + d
        if 0 <= p < len(order):
            c = order[p]
            if abs(period[c] - period[seed]) <= window:
                idx.append(c)
    if len(idx) >= 2:
        a, _ = np.polyfit(lines[idx], period[idx], 1)
        return a
    return 0.0


def _grow(seed, direction, members, group_id, gid,
          period, lines, n_cycles, order, rank, delta, max_jumps, slope0):
    """Extend the group from the seed in one direction of the 'lines' axis."""
    pos = rank[seed]
    jumps = 0
    n = len(order)
    while True:
        pos += direction
        if pos < 0 or pos >= n:
            break
        cand = order[pos]

        # candidate already assigned -> jump (holes tolerated up to max_jumps)
        if group_id[cand] != -1:
            jumps += 1
            if jumps > max_jumps:
                break
            continue

        # expected period on the line of the group, at this value of 'lines'
        lm = lines[members]
        pm = period[members]
        cm = n_cycles[members]
        if len(members) >= 2:
            a, b = np.polyfit(lm, pm, 1)         # group fit
            period_pred = a * lines[cand] + b
        else:
            period_pred = pm[0] + slope0 * (lines[cand] - lm[0])  # seed alone

        if len(members) >= 2:
            a, b = np.polyfit(lm, cm, 1)         # group fit
            cycles_pred = a * lines[cand] + b
        else:
            cycles_pred = cm[0] + slope0 * (lines[cand] - lm[0])  # seed alone

        if abs(period[cand] - period_pred) <= delta: # and abs(n_cycles[cand] - cycles_pred) <= delta:
            group_id[cand] = gid
            members.append(cand)
            jumps = 0
        else:
            jumps += 1
            if jumps > max_jumps:
                break

    


def find_groups(period, magnitude, lines, n_cycles, delta, max_jumps=2, n_groups=None, min_cycles=20):
    """
    Group the points (period vs lines) aligned on a same straight line.

    Parameters
    ----------
    period, magnitude, lines : array-like (same size)
    delta : float
        Tolerance in 'period' around the line of the group.
    max_jumps : int
        Number of missing/rejected 'lines' tolerated before stopping the extension
        of a group (handles the holes in a crest).

    Returns
    -------
    group_id : np.ndarray of integers (same size)
        Group identifier (>= 0) of each point. The seed of a group (its point of
        maximum magnitude) is guaranteed to be the maximum of the group.
    """
    period    = np.asarray(period, dtype=float)
    magnitude = np.asarray(magnitude, dtype=float)
    lines     = np.asarray(lines, dtype=float)
    n_cycles  = np.asarray(n_cycles, dtype=float)
    n = len(period)

    group_id = np.full(n, -1, dtype=int)         # -1 = free, -2 = rejected
    order = np.argsort(lines)
    rank = np.empty(n, dtype=int); rank[order] = np.arange(n)

    gid = 0
    while np.any(group_id == -1):
        if n_groups is not None and gid >= n_groups:
            break
        free = np.where(group_id == -1)[0]
        seed = free[np.argmax(magnitude[free])]

        members = [seed]
        group_id[seed] = gid
        slope0 = _initial_slope(seed, period, lines, order, rank, delta)
        _grow(seed, +1, members, group_id, gid,
              period, lines, n_cycles, order, rank, delta, max_jumps, slope0)
        _grow(seed, -1, members, group_id, gid,
              period, lines, n_cycles, order, rank, delta, max_jumps, slope0)

        # validation: mean number of cycles of the group > min_cycles
        if n_cycles[members].mean() > min_cycles:
            gid += 1                              # group validated
        else:
            # print(members, n_cycles[members].mean())
            group_id[members] = -2                # rejected, excluded from the seeds

    group_id[group_id == -2] = -1                 # the rejected ones -> unclassified
        
    # --- relabelling: group 0 = best mean magnitude per point ---
    ids = [g for g in np.unique(group_id) if g != -1]

    ## sort by magnitude
    # sizes = {g: magnitude[group_id == g].mean() for g in ids}
    ## sort by number of points in the group:
    sizes = {g: len(np.where(group_id == g)[0]) for g in ids}

    group_order = sorted(ids, key=lambda g: sizes[g], reverse=True)
    remap = {old: new for new, old in enumerate(group_order)}

    new_id = np.full_like(group_id, -1)
    for old, new in remap.items():
        new_id[group_id == old] = new
    group_id = new_id

    return group_id


rows_arr   = np.array([r[0] for r in results], dtype=float)
phases_arr  = np.array([r[2] for r in results], dtype=float)
mag_arr = np.array([r[3] for r in results], dtype=float)

periods_arr = np.array([r[1] for r in results], dtype=float)

period, magnitude, lines = periods_arr, mag_arr, rows_arr

gid = find_groups(period, magnitude, lines, n_cycles, delta=1, n_groups=5)
print("Groups found :", gid.max() + 1)


fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# unclassified points (group_id == -1)
m = gid == -1
if m.any():
    axes[0].scatter(lines[m], period[m], s=25, c="lightgray", label="non classified")

# groups (starting from 0, ignoring the -1)
for g in range(gid.max() + 1):
    m = gid == g
    if not m.any():
        continue
    axes[0].scatter(lines[m], period[m], s=40, label=f"group {g}")
    imax = np.argmax(np.where(m, magnitude, -np.inf))
    axes[0].scatter(lines[imax], period[imax], s=180,
                facecolors="none", edgecolors="k", linewidths=1.5)
axes[0].set_xlabel("lines"); axes[0].set_ylabel("period"); axes[0].legend()
axes[0].set_title("groups (circle = max magnitude)")


axes[1].scatter(rows_arr, periods_arr, s=4, alpha=0.5, c=[res[3] for res in results], cmap='viridis',)

plt.tight_layout()
plt.show()

In [ ]:
ev = group_evidence(gid, row_specs, rows_arr, periods_arr,
                        reduced_img_gray, n_candidate_rows=len(row_indices))
conf, debug = ruler_confidence(ev)

print(conf, debug)


In [ ]:
# update of the line
indices = np.where(gid == 0)[0]
max_line_idx = int(indices.mean())
mean_period = np.asarray(periods_arr)[indices].mean()
mean_phase = np.asarray(phases_arr)[indices].mean()
print(max_line_idx, mean_period, mean_phase)

if len(np.unique(gid)) > 2:
    # other possible location
    indices_second = np.where(gid == 1)[0]
    max_line_idx_second = int(indices_second.mean())
    mean_period_second = np.asarray(periods_arr)[indices_second].mean()
    mean_phase_second = np.asarray(phases_arr)[indices_second].mean()
    print(max_line_idx_second, mean_period_second, mean_phase_second)
else:
    indices_second = indices
    max_line_idx_second = max_line_idx
    mean_period_second = mean_period
    mean_phase_second = mean_phase
    print("no other possible group")

In [ ]:
T_median = mean_period * RATIO
T_corrected = mean_period / np.cos(mean_phase) * RATIO

px_per_mm = T_corrected / GRADUATION_MM 
mm_per_px = GRADUATION_MM / T_corrected


print(f"Ruler gradation : {GRADUATION_MM:.2f} mm")
print(f"Raw period (X-axis): {T_median:.3f} px")
print(f"Inclination angle: {theta_deg:.3f} °")
print(f"Corrected period: {T_corrected:.3f} px")

print(f"px / mm: {px_per_mm:.4f} px/mm")
print(f"mm / px: {mm_per_px:.6f} mm/px")


In [ ]:

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(base_img_color)
ax1.set_title("Analysed image", fontsize=12)
ax1.axis('off')

info = (f"  mm/px = {mm_per_px:.5f}  |  px/mm = {px_per_mm:.3f}  "
        f"|  angle = {theta_deg:.2f}°  ")
ax1.set_xlabel(info, fontsize=10, labelpad=6)
ax1.xaxis.label.set_color('darkgreen')

modif_reduced_img_gray = reduced_img_gray[..., np.newaxis]
modif_reduced_img_gray = np.concatenate([modif_reduced_img_gray, modif_reduced_img_gray, modif_reduced_img_gray], axis = 2)
print(modif_reduced_img_gray.shape)
print(max_line_idx)
modif_reduced_img_gray[max_line_idx_second, :, :] = [255, 100, 100]
modif_reduced_img_gray[max_line_idx, :, :] = [255, 0, 0]
ax4 = fig.add_subplot(gs[0, 1])
ax4.imshow(modif_reduced_img_gray)
ax4.set_title("Ruler zone", fontsize=12)
ax4.axis('off')

info = (f"  mm/px = {mm_per_px:.5f}  |  px/mm = {px_per_mm:.3f}  "
        f"|  angle = {theta_deg:.2f}°  ")
ax4.set_xlabel(info, fontsize=10, labelpad=6)
ax4.xaxis.label.set_color('darkgreen')

ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(periods_arr * RATIO, bins=40, color='steelblue', edgecolor='white', lw=0.4)
ax2.axvline(T_median, color='red', ls='--', lw=2,
            label=f'Median = {T_median:.2f} px')
ax2.axvline(T_corrected, color='darkgreen', ls='-', lw=2,
            label=f'Corrected = {T_corrected:.2f} px')
ax2.set_title("FFT period distribution", fontsize=11)
ax2.set_xlabel("Period (px)")
ax2.set_ylabel("Number of lines")
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

ax3 = fig.add_subplot(gs[1, 1])
ax3.axis('off')
summary_lines = [
    "SUMMARY",
    "",
    f"Ruler graduation  : {GRADUATION_MM} mm",
    f"Analyzed lines    : {len(row_indices)}",
    f"Valid lines       : {len(results)}",
    "",
    f"Raw period        : {T_median:.3f} px",
    f"Inclination angle : {theta_deg:.3f}°",
    f"Corrected period  : {T_corrected:.3f} px",
    "",
    f"  px / mm       : {px_per_mm:.4f}",
    f"  mm / px       : {mm_per_px:.6f}",
]
summary_text = "\n".join(summary_lines)
ax3.text(0.05, 0.95, summary_text,
         transform=ax3.transAxes,
         fontsize=10, verticalalignment='top', family='monospace',
         bbox=dict(boxstyle='round', facecolor='#f0f8e8', alpha=0.8))

plt.suptitle("FFT Analysis — Ruler Calibration",
             fontsize=13, fontweight='bold', y=1.01)
plt.show()
